# 03 — Generation Agent: Dense GRPO from SFT

Inputs: the **trained SFT adapter checkpoint** from Notebook 02, training questions/schema/gold SQL, reasoning bank/evidence and SQLite databases. Model responses are sampled independently of gold labels; gold is available only to reward calculation. Output: optimized adapter with optimizer/RNG state. Use 3B first; 7B is an optional memory-sensitive experiment, not a promised T4 configuration.


In [ ]:
import os
import sys
from pathlib import Path

# Attach a repository snapshot as a Kaggle Dataset, or set KAGGLE_REPO_DIR.
_candidates = [Path(os.environ.get("KAGGLE_REPO_DIR", ".")), Path.cwd(), Path.cwd().parent]
_candidates += list(Path("/kaggle/input").glob("*/Optimize-Mars-Sl")) if Path("/kaggle/input").exists() else []
REPO_DIR = next((p.resolve() for p in _candidates if (p / "src").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Set KAGGLE_REPO_DIR to the attached repository directory containing src/")
sys.path.insert(0, str(REPO_DIR))
from src.training.kaggle import KagglePaths, install_dependencies, environment_report
install_dependencies(REPO_DIR)  # Only runs pip when INSTALL_DEPS=1.
paths = KagglePaths.from_env()
SMOKE = paths.smoke
print({"mode": "smoke (NOT a benchmark)" if SMOKE else "train", "data": str(paths.data_dir), "device": paths.device})


In [ ]:
PHASE = 'grpo_3b'
import json
OUTPUT_DIR = paths.output(PHASE)
environment = environment_report(OUTPUT_DIR)
MAX_STEPS = int(os.environ.get("MAX_STEPS", "100"))
MAX_LENGTH = int(os.environ.get("MAX_LENGTH", "512"))
SEED = int(os.environ.get("SEED", "42"))
SAVE_STEPS = int(os.environ.get("SAVE_STEPS", "25"))
RESUME = os.environ.get("RESUME_FROM_CHECKPOINT") or None  # "latest" or attached checkpoint path
print({"output": OUTPUT_DIR, "resume_from_checkpoint": RESUME, "gpus": environment["gpus"]})


Set `SFT_CHECKPOINT` to an attached Notebook 02 checkpoint directory, `GRPO_DATA` to training JSONL and `DB_DIR` to databases. Begin with `GROUP_SIZE=2`, `MAX_NEW_TOKENS=128`. Constant-reward groups carry zero advantages and provide no policy gradient; logs expose this rather than inventing reward variance.

In [ ]:
from src.training.train_grpo_generator import run_grpo_training
GRPO_DATA = paths.input("GRPO_DATA", "mini_bird.jsonl" if SMOKE else "grpo_train.jsonl")
DB_DIR = paths.input("DB_DIR", "." if SMOKE else "databases")
SFT_CHECKPOINT = os.environ.get("SFT_CHECKPOINT")
summary = run_grpo_training(data_path=GRPO_DATA, db_dir=DB_DIR,
    output_dir=OUTPUT_DIR, dry_run=SMOKE, sft_checkpoint=SFT_CHECKPOINT,
    device=paths.device, max_steps=MAX_STEPS, max_length=MAX_LENGTH, save_steps=SAVE_STEPS,
    seed=SEED, num_rollouts=int(os.environ.get("GROUP_SIZE", "2")),
    max_new_tokens=int(os.environ.get("MAX_NEW_TOKENS", "128")),
    gradient_accumulation_steps=int(os.environ.get("GRAD_ACCUM", "8")),
    resume_from_checkpoint=RESUME)
print(json.dumps(summary, indent=2))


## Artifact handoff and validation

The last cell returns artifact paths. Download `/kaggle/working/optimize-mars` after a run or save a notebook version with outputs, then attach those outputs to the downstream notebook as a Kaggle Dataset. Runtime storage is ephemeral; a local checkpoint alone does not survive session deletion. Export before the session ends. Set `RESUME_FROM_CHECKPOINT` to an attached complete checkpoint, or `latest` for the current output directory. No checkpoints are automatically deleted.

Use `PIPELINE_MODE=smoke` for offline wiring checks; this mode does not train pretrained models or measure model quality. Default `train` requires real datasets, dependencies, GPU and the input artifacts listed above. Settings target **one 16 GB T4**, sequential rollouts and short sequences; T4 x2 does not provide a shared 32 GB memory pool. Reduce `MAX_LENGTH`, rollout length or accumulation if needed. No full GPU run or Kaggle benchmark is claimed by this notebook. See `KAGGLE.md` for data contracts, independent account handoffs and ablations.
